# S3 Traffic Generator (AWS MCP + Colab)

Generate S3 object traffic for GuardrailStudio via the [AWS API MCP Server](https://github.com/awslabs/mcp/tree/main/src/aws-api-mcp-server) (`call_aws` → `aws s3api` CRUD).

**Setup (Colab 🔑 Secrets):**
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_REGION` *(optional, default `us-east-1`)*

**Files:** run with the three helpers in `colab/` (`aws_credentials.py`, `mcp_s3_client.py`, `traffic_generator.py`) in the same directory, or upload them when prompted.

Use a scoped IAM user with `sts:GetCallerIdentity` and S3 permissions on the target bucket.

In [ ]:
%pip install -q awslabs.aws-api-mcp-server mcp

In [ ]:
import sys
from pathlib import Path

MODULE_NAMES = ("aws_credentials.py", "mcp_s3_client.py", "traffic_generator.py")


def _find_module_dir() -> Path | None:
    candidates = [
        Path.cwd(),
        Path.cwd() / "colab",
        Path("/content/traffic-generator/colab"),
        Path("/content/colab"),
    ]
    for directory in candidates:
        if all((directory / name).exists() for name in MODULE_NAMES):
            return directory.resolve()
    return None


module_dir = _find_module_dir()
if module_dir is None:
    print("Upload the helper modules from colab/:")
    for name in MODULE_NAMES:
        print(f"  • {name}")
    from google.colab import files

    uploaded = files.upload()
    module_dir = Path("/content/colab")
    module_dir.mkdir(parents=True, exist_ok=True)
    for name, content in uploaded.items():
        (module_dir / name).write_bytes(content)

    missing = [name for name in MODULE_NAMES if not (module_dir / name).exists()]
    if missing:
        raise SystemExit(f"Still missing: {', '.join(missing)}")

sys.path.insert(0, str(module_dir))
print(f"Using modules from: {module_dir}")

In [ ]:
from aws_credentials import CredentialError, mask_access_key, resolve_aws_credentials
from mcp_s3_client import AwsMcpS3Client
from traffic_generator import run_traffic_loop, verify_access

BUCKET = "synapse6-demo-bucket"  # @param {type:"string"}
CYCLES = 20  # @param {type:"integer"}
MIN_INTERVAL_SEC = 3.0  # @param {type:"number"}
MAX_INTERVAL_SEC = 12.0  # @param {type:"number"}
READ_ONLY = False  # @param {type:"boolean"}

try:
    creds = resolve_aws_credentials()
    print(f"Using {mask_access_key(creds.access_key_id)} in {creds.region}")
except CredentialError as exc:
    raise SystemExit(exc) from exc

mcp = AwsMcpS3Client(creds, read_only=READ_ONLY)
stats = None
try:
    await mcp.__aenter__()
    await verify_access(mcp, BUCKET)
    print("Smoke test OK")
    stats = await run_traffic_loop(
        bucket=BUCKET,
        credentials=creds,
        client=mcp,
        cycles=CYCLES,
        min_interval_sec=MIN_INTERVAL_SEC,
        max_interval_sec=MAX_INTERVAL_SEC,
        read_only=READ_ONLY,
        verify=False,
    )
finally:
    await mcp.aclose()

stats

Events reach GuardrailStudio when the bucket has SNS/EventBridge → `/v1/events/webhook/s3`.